# Portfolio-opdracht 2: Voorspel verhuuraantallen (CityBike)

<a id="top"></a>

| Veld | Invullen |
|---|---|
| **Teamnaam (Kaggle)** | *TODO* |
| **Competitie** | [Voorspel verhuuraantallen 2026](https://www.kaggle.com/competitions/voorspel-verhuuraantallen-2026) |
| **Datum / versie** | *TODO* |

| Teamlid | Studentnummer | Kaggle-gebruikersnaam |
|---|---|---|
| *TODO naam 1* | *TODO* | *TODO* |
| *TODO naam 2* | *TODO* | *TODO* |
| *TODO naam 3* | *TODO* | *TODO* |
| *TODO naam 4* | *TODO* | *TODO* |

> **Hoe gebruik je dit template?**
> - Alles wat met **TODO** is gemarkeerd moet je zelf invullen of aanpassen.
> - Codecellen zijn werkende *startpunten*: draai ze, begrijp ze, breid ze
>   uit en pas ze aan. Alle teamleden moeten elke cel kunnen toelichten.
> - Blokken die beginnen met `> **Instructie**` zijn aanwijzingen voor
>   jullie zelf. **Verwijder ze voor het inleveren.**
> - Schrijf alle uitleg in **eigen woorden** en verwijs naar bronnen
>   (APA, zie hoofdstuk 8). Gebruikte GenAI-prompts komen in de
>   GAI-tabel in hoofdstuk 8.

## Inhoudsopgave

1. [Inleiding](#h1)
2. [Setup, configuratie en data inladen](#h2)
3. [Opdracht 1: Exploratieve Data Analyse](#h3)
4. [Opdracht 2: Time series feature engineering](#h4)
5. [Opdracht 3: Modelleren](#h5)
6. [Bevindingen](#h6)
7. [Opdracht 4: Conclusie en aanbevelingen](#h7)
8. [Referenties en GenAI-gebruik](#h8)
9. [Inleverchecklist](#h9)

> **Instructie: harde eisen (anders wordt het notebook niet nagekeken)**
>
> - Één net, duidelijk gestructureerd notebook; genummerde hoofdstukken en
>   paragrafen; markdown voor tekst, code voor code.
> - Zoveel mogelijk zelfgedefinieerde functies, bij voorkeur OOP.
> - Code volgt PEP8 en heeft consistente comments (naamgeving en stijl
>   uniform in het hele notebook).
> - Minstens één Kaggle-submission, met een **RMSE van maximaal 80**, en
>   uitsluitend afkomstig van jullie **eigen getrainde modellen**.
> - Bronnen voor tekst, figuren en code in APA-stijl; bij GenAI als bron
>   óók een niet-GenAI bron. Maximaal 5% letterlijke tekst en 25%
>   letterlijke code.

<a id="h1"></a>
# 1. Inleiding

[Terug naar boven](#top)

> **Instructie:** schrijf deze inleiding zelf; houd het kort (± 1 pagina).

**TODO – Bedrijfscontext.** Wie is CityBike en waarom is het belangrijk om
het aantal verhuurde fietsen per uur te voorspellen? (Denk aan te weinig
fietsen = gemiste verhuur; te veel fietsen op één plek = inefficiënt gebruik.)

**TODO – Probleemstelling en doel.** Formuleer de onderzoeksvraag en het
type machine-learningprobleem (regressie op tijdreeksdata, doelvariabele
`cnt`, evaluatiemetriek RMSE).

**TODO – Aanpak.** Beschrijf in een paar zinnen de stappen: EDA, time series
feature engineering, modelleren (reguliere modellen, tijdseriemodellen,
hybride model), evaluatie met tijdsgebaseerde cross-validation, Kaggle.

**TODO – Leeswijzer.** Welke hoofdstukken horen bij welke opdracht?

<a id="h2"></a>
# 2. Setup, configuratie en data inladen

[Terug naar boven](#top)

## 2.1 Imports

> Benodigde pakketten (indien nog niet geïnstalleerd):
> `pip install pandas numpy scipy scikit-learn statsmodels matplotlib seaborn`

In [ ]:
# Standaardbibliotheken
import warnings
from dataclasses import dataclass, field, replace
from datetime import datetime
from pathlib import Path

# Data en visualisatie
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

# Statistiek en tijdreeksen
from scipy import signal, stats
from scipy.fft import rfft, rfftfreq
from scipy.stats import loguniform, randint
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, kpss

# Machine learning
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (HistGradientBoostingRegressor,
                              RandomForestRegressor)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 4)

## 2.2 Configuratie

Alle instellingen staan op één plek, zodat het notebook reproduceerbaar is
(vaste `random_state`) en paden/bestandsnamen makkelijk aan te passen zijn.

In [ ]:
@dataclass(frozen=True)
class Config:
    """Centrale instellingen van het project."""

    data_dir: Path = Path("data")            # map met de Kaggle-bestanden
    submission_dir: Path = Path("submissions")
    train_file: str = "train.csv"
    test_file: str = "test.csv"
    sample_file: str = "sample_submission.csv"
    date_col: str = "date_hour"
    target: str = "cnt"
    random_state: int = 42
    n_splits: int = 5                        # folds voor cross-validation
    cv_test_size: int = 24 * 19              # 19 dagen = lengte testperiode
    min_rmse: float = 80.0                   # eis Kaggle-score


CFG = Config()
CFG.submission_dir.mkdir(exist_ok=True)

# Referentiepunt voor trend en Fourier-termen (begin van de trainingsdata)
EPOCH = pd.Timestamp("2011-01-01")

## 2.3 Hulpfuncties en experiment-logboek

- `rmse`: de evaluatiemetriek van de competitie.
- `make_cv` / `cross_val_rmse`: **tijdsgebaseerde** cross-validation
  (expanding window). Waarom geen gewone shuffled K-fold? Zie §5.1.
- `ExperimentLog`: legt elk experiment vast (model, featureset, CV-score,
  Kaggle-score). Hiermee kunnen jullie later laten zien **hoe** jullie tot
  de uiteindelijke feature- en modelkeuze zijn gekomen.

In [ ]:
def rmse(y_true, y_pred):
    """Bereken de Root Mean Squared Error (metriek van de competitie)."""
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def make_cv(n_splits=CFG.n_splits, test_size=CFG.cv_test_size):
    """Maak een tijdsgebaseerde cross-validation (expanding window)."""
    return TimeSeriesSplit(n_splits=n_splits, test_size=test_size)


def cross_val_rmse(model, X, y, cv=None):
    """Bereken de RMSE per fold met tijdsgebaseerde cross-validation.

    Voorspellingen worden op 0 afgekapt omdat aantallen niet negatief
    kunnen zijn.
    """
    cv = make_cv() if cv is None else cv
    scores = []
    for train_idx, val_idx in cv.split(X):
        fitted = clone(model).fit(X.iloc[train_idx], y.iloc[train_idx])
        pred = np.clip(fitted.predict(X.iloc[val_idx]), 0, None)
        scores.append(rmse(y.iloc[val_idx], pred))
    return np.array(scores)


def ts_cv_rmse(fit_predict, y, cv=None):
    """Cross-validation voor tijdseriemodellen.

    `fit_predict(train_idx, val_idx)` moet de voorspellingen voor de
    validatieperiode teruggeven (zelfde lengte als `val_idx`).
    """
    cv = make_cv() if cv is None else cv
    scores = []
    for train_idx, val_idx in cv.split(y):
        pred = np.clip(fit_predict(train_idx, val_idx), 0, None)
        scores.append(rmse(y.iloc[val_idx], pred))
    return np.array(scores)


class ExperimentLog:
    """Houdt alle experimenten bij: model, features, CV- en Kaggle-score."""

    COLUMNS = ["timestamp", "experiment", "model", "feature_set",
               "cv_rmse", "cv_std", "params", "kaggle_rmse", "notes"]

    def __init__(self):
        self.records = []

    def add(self, experiment, model, feature_set, cv_rmse, cv_std=np.nan,
            params=None, notes=""):
        """Voeg een experiment toe (of overschrijf een bestaande naam)."""
        self.records = [r for r in self.records
                        if r["experiment"] != experiment]
        self.records.append({
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
            "experiment": experiment, "model": model,
            "feature_set": feature_set, "cv_rmse": float(cv_rmse),
            "cv_std": float(cv_std), "params": params or {},
            "kaggle_rmse": np.nan, "notes": notes})

    def set_kaggle_score(self, experiment, score):
        """Koppel een Kaggle-score aan een eerder gelogd experiment."""
        for record in self.records:
            if record["experiment"] == experiment:
                record["kaggle_rmse"] = float(score)

    def to_frame(self):
        """Geef het logboek als DataFrame."""
        return pd.DataFrame(self.records, columns=self.COLUMNS)

    def save(self, path="experiment_log.csv"):
        """Sla het logboek op als CSV (back-up)."""
        self.to_frame().to_csv(path, index=False)


LOG = ExperimentLog()

## 2.4 Data inladen

> **Instructie:** zet `train.csv`, `test.csv` en `sample_submission.csv` in
> de map `data/` naast dit notebook (of pas `Config.data_dir` aan).

In [ ]:
class DataLoader:
    """Leest de Kaggle-bestanden in en zet de datumkolom om."""

    def __init__(self, config):
        self.config = config

    def _read(self, filename):
        """Lees één csv-bestand en parse de datumkolom."""
        df = pd.read_csv(self.config.data_dir / filename)
        col = self.config.date_col
        df[col] = pd.to_datetime(df[col])
        return df

    def load_all(self):
        """Geef (train, test, sample_submission) terug."""
        return (self._read(self.config.train_file),
                self._read(self.config.test_file),
                self._read(self.config.sample_file))


train_raw, test_raw, sample_raw = DataLoader(CFG).load_all()
for name, frame in [("train", train_raw), ("test", test_raw),
                    ("sample_submission", sample_raw)]:
    first, last = frame[CFG.date_col].min(), frame[CFG.date_col].max()
    print(f"{name:18s} {frame.shape}  {first} -> {last}")
display(train_raw.head())

<a id="h3"></a>
# 3. Opdracht 1: Exploratieve Data Analyse

[Terug naar boven](#top)

> **Instructie: vereisten Opdracht 1**
> - Teamnaam, namen en Kaggle-gebruikersnamen staan bovenaan (ingevuld?).
> - Datatypes en basisstatistieken van iedere kolom.
> - Tijdserie-elementen met passende visualisaties.
> - Relaties tussen onafhankelijke variabelen en de target, met visualisaties.
> - Belangrijkste bevindingen beschrijven.
> - Data opschonen waar nodig.

## 3.1 Datatypes en basisstatistieken

In [ ]:
def describe_columns(df):
    """Geef per kolom datatype, ontbrekende waarden en basisstatistieken."""
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(),
    })
    numeric = df.select_dtypes("number").describe().T
    return summary.join(numeric[["mean", "std", "min", "50%", "max"]])


display(describe_columns(train_raw))
print("Kolommen test:", list(test_raw.columns))
print("Zelfde features als train:",
      set(test_raw.columns) == set(train_raw.columns) - {CFG.target})

> **TODO (team):** beschrijf per kolom wat het datatype is en of dat klopt.
> Let op: de kolommen `holiday` en `weathersit` zijn **categorisch** (ook al
> staan ze als getal in de data). Controleer de codering (0/1? 1-4?) en pas
> `DataCleaner.clean` hieronder aan als de codering afwijkt. De kolommen
> `temp`, `atemp`, `hum`, `windspeed` zijn genormaliseerd (verwacht 0-1).

## 3.2 Datakwaliteit en opschoning

Controleer onder meer: dubbele tijdstippen, ontbrekende uren in de reeks,
ontbrekende waarden, onmogelijke waarden (buiten 0-1) en verdachte waarden
(bijv. luchtvochtigheid van precies 0).

> **Let op:** in de **testset mogen nooit rijen verwijderd worden**; Kaggle
> verwacht een voorspelling voor elk tijdstip. Daar imputeer je alleen.

In [ ]:
class DataCleaner:
    """Controleert en schoont de data op."""

    NORMALISED = ["temp", "atemp", "hum", "windspeed"]
    WEATHER = ["weathersit", "temp", "atemp", "hum", "windspeed"]

    def __init__(self, config):
        self.config = config

    def report(self, df):
        """Geef een overzicht van mogelijke datakwaliteitsproblemen."""
        dates = pd.DatetimeIndex(df[self.config.date_col])
        full_range = pd.date_range(dates.min(), dates.max(), freq="h")
        missing_values = df.isna().sum()
        return {
            "n_rows": len(df),
            "duplicate_timestamps": int(dates.duplicated().sum()),
            "missing_hours_in_range": len(full_range.difference(dates)),
            "missing_values": missing_values[missing_values > 0].to_dict(),
            "outside_0_1": {c: int(((df[c] < 0) | (df[c] > 1)).sum())
                            for c in self.NORMALISED},
            "humidity_exactly_0": int((df["hum"] == 0).sum()),
        }

    def clean(self, df, is_train=True):
        """Schoon de data op en geef een DataFrame met datum als index."""
        col, target = self.config.date_col, self.config.target
        out = df.sort_values(col)
        if is_train:  # in de testset nooit rijen verwijderen
            out = out.drop_duplicates(subset=col, keep="first")
        out = out.set_index(col).copy()

        # Onmogelijke waarden worden NaN en later geïmputeerd
        for column in self.NORMALISED:
            invalid = (out[column] < 0) | (out[column] > 1)
            out.loc[invalid, column] = np.nan
        # TODO: onderbouw of luchtvochtigheid = 0 echt een foutieve meting is
        out.loc[out["hum"] == 0, "hum"] = np.nan

        if is_train:  # alleen trainrijen met een geldige target behouden
            out = out[out[target].notna() & (out[target] >= 0)]

        # Imputeer ontbrekende weerswaarden op basis van de tijd
        out[self.WEATHER] = out[self.WEATHER].interpolate(
            method="time", limit_direction="both")
        out["weathersit"] = out["weathersit"].round().astype(int)
        # TODO: pas aan als holiday niet als 0/1 gecodeerd is
        out["holiday"] = out["holiday"].fillna(0).astype(int)
        return out

    def to_regular_hourly(self, df):
        """Maak een regelmatige uurindex (nodig voor tijdseriemodellen).

        Ontbrekende uren krijgen NaN voor de target; weer wordt
        geïnterpoleerd en `holiday` wordt per dag doorgezet.
        """
        reg = df.asfreq("h")
        reg[self.WEATHER] = reg[self.WEATHER].interpolate(
            method="time", limit_direction="both")
        reg["weathersit"] = reg["weathersit"].round().astype(int)
        day = reg.index.normalize()
        reg["holiday"] = (reg["holiday"].groupby(day).transform("max")
                          .fillna(0).astype(int))
        return reg


cleaner = DataCleaner(CFG)
print("TRAIN:", cleaner.report(train_raw))
print("TEST: ", cleaner.report(test_raw))

> **TODO (team):** beschrijf welke problemen je vond en **onderbouw** elke
> opschoonstap (verwijderen, imputeren, laten staan). Wat is de impact op
> het aantal rijen? Zijn ontbrekende uren waarschijnlijk uren met 0
> verhuurde fietsen (bijv. 's nachts), of echt ontbrekende data? Dat bepaalt
> hoe je ze invult in de regelmatige reeks hieronder.

In [ ]:
train_clean = cleaner.clean(train_raw, is_train=True)
test_clean = cleaner.clean(test_raw, is_train=False)
print(f"train: {len(train_raw)} -> {len(train_clean)} rijen")
print(f"test : {len(test_raw)} -> {len(test_clean)} rijen")

# Gecombineerde (feature-)data zonder target: nodig voor rolling features
# die tijdens de testperiode context uit het einde van de trainperiode nodig
# hebben. Er wordt hierbij GEEN informatie uit de target gebruikt.
full_clean = pd.concat([train_clean.drop(columns=CFG.target),
                        test_clean]).sort_index()

# Regelmatige uurreeks (voor SARIMAX / Holt-Winters / Fourier-analyse)
train_end = train_clean.index.max()
full_reg = cleaner.to_regular_hourly(
    pd.concat([train_clean, test_clean]).sort_index())
# TODO: onderbouw de vulstrategie voor ontbrekende uren (0 of interpoleren)
train_part = full_reg.index <= train_end
full_reg.loc[train_part, CFG.target] = (
    full_reg.loc[train_part, CFG.target].fillna(0))
y_reg = full_reg.loc[train_part, CFG.target]
print("Regelmatige trainreeks:", y_reg.shape,
      "| nieuw ingevulde uren:", len(y_reg) - len(train_clean))

## 3.3 Tijdserie-EDA

Onderzoek hoe het aantal verhuurde fietsen verandert **over de tijd** en
binnen **dag, week, maand en jaar**.

In [ ]:
def add_calendar_columns(df):
    """Voeg kalenderkolommen toe voor verkennende analyse."""
    out = df.copy()
    out["hour"] = out.index.hour
    out["dayofweek"] = out.index.dayofweek
    out["month"] = out.index.month
    out["year"] = out.index.year
    out["is_weekend"] = (out["dayofweek"] >= 5).astype(int)
    out["workingday"] = ((out["dayofweek"] < 5)
                         & (out["holiday"] == 0)).astype(int)
    return out


eda = add_calendar_columns(train_clean)
daily = eda[CFG.target].resample("D").mean().interpolate()

fig, ax = plt.subplots()
daily.plot(ax=ax, alpha=0.35, label="dagelijks gemiddelde")
daily.rolling(7, center=True).mean().plot(ax=ax, label="7-daags gemiddelde")
daily.rolling(30, center=True).mean().plot(ax=ax,
                                           label="30-daags gemiddelde")
ax.set(title="Gemiddeld aantal verhuurde fietsen per uur, per dag",
       xlabel="Datum", ylabel="cnt")
ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Uurprofiel: werkdag versus niet-werkdag
sns.lineplot(data=eda, x="hour", y=CFG.target, hue="workingday",
             estimator="mean", errorbar=("pi", 50), ax=axes[0, 0])
axes[0, 0].set_title("Uurprofiel (werkdag vs. geen werkdag)")

# Weekdagpatroon
sns.barplot(data=eda, x="dayofweek", y=CFG.target, errorbar=None,
            ax=axes[0, 1])
axes[0, 1].set_title("Gemiddelde per weekdag (0 = maandag)")

# Seizoenspatroon per jaar
sns.lineplot(data=eda, x="month", y=CFG.target, hue="year", marker="o",
             errorbar=None, palette="tab10", ax=axes[1, 0])
axes[1, 0].set_title("Gemiddelde per maand, per jaar")

# Verdeling van de target
sns.histplot(eda[CFG.target], bins=50, ax=axes[1, 1])
axes[1, 1].set_title("Verdeling van cnt")
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: uur x weekdag
pivot = eda.pivot_table(index="hour", columns="dayofweek",
                        values=CFG.target, aggfunc="mean")
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(pivot, cmap="viridis", ax=ax)
ax.set(title="Gemiddelde cnt per uur en weekdag", xlabel="Weekdag")
plt.show()

# Seizoensdecompositie van het dagelijkse gemiddelde (weekpatroon)
stl_result = STL(daily, period=7, robust=True).fit()
fig = stl_result.plot()
fig.set_size_inches(12, 8)
plt.show()

> **TODO (team):** beschrijf wat je ziet. Denk aan: dagpatroon (pieken
> tijdens woon-werkverkeer?), verschil werkdag/weekend, jaarlijks seizoen,
> **groei van 2011 naar 2012** (trend), uitschieters, en dalen rond feestdagen
> of slecht weer. Welke patronen verwacht je dat een model moet leren?

## 3.4 Relaties tussen de variabelen en de target

In [ ]:
def plot_target_relations(df, features, target):
    """Maak hexbin-plots van numerieke features tegen de target."""
    fig, axes = plt.subplots(1, len(features), sharey=True,
                             figsize=(4.5 * len(features), 4))
    for ax, feature in zip(axes, features):
        ax.hexbin(df[feature], df[target], gridsize=30, cmap="viridis",
                  mincnt=1)
        ax.set_xlabel(feature)
    axes[0].set_ylabel(target)
    plt.tight_layout()
    plt.show()


numeric_features = ["temp", "atemp", "hum", "windspeed"]
plot_target_relations(eda, numeric_features, CFG.target)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=eda, x="weathersit", y=CFG.target, ax=axes[0])
axes[0].set_title("cnt per weersituatie")
sns.boxplot(data=eda, x="holiday", y=CFG.target, ax=axes[1])
axes[1].set_title("cnt op vakantiedag (1) of niet (0)")
plt.show()

In [ ]:
# Correlaties (Spearman: ook geschikt voor niet-lineaire monotone relaties)
corr_cols = numeric_features + ["weathersit", "holiday", "hour",
                                "dayofweek", "month", CFG.target]
corr = eda[corr_cols].corr(method="spearman")
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Spearman-correlatiematrix")
plt.show()

> **TODO (team):** bespreek per variabele de relatie met `cnt` (richting,
> sterkte, lineair of niet-lineair?) én relaties **tussen** de features
> (bijv. `temp` en `atemp`: multicollineariteit, wat betekent dat voor
> lineaire modellen?). Let op: correlatie ≠ causaliteit, en `hour` is
> cyclisch (uur 23 ligt dicht bij uur 0), dus een rechte-lijncorrelatie met
> `hour` zegt weinig.

## 3.5 Belangrijkste bevindingen van de EDA

> **TODO (team):** vat samen in 5-8 puntsgewijze, onderbouwde bevindingen
> (verwijs naar de figuren hierboven). Leg ook uit welke bevindingen jullie
> aanpak in hoofdstuk 4 en 5 sturen (bijv. "sterk dagpatroon, dus Fourier-
> termen en interactie met werkdag").

1. *TODO*
2. *TODO*
3. *TODO*

<a id="h4"></a>
# 4. Opdracht 2: Time series feature engineering

[Terug naar boven](#top)

> **Instructie: vereisten Opdracht 2** — duidelijk **onderzocht én
> uitgelegd**:
> 1. Seizoenspatronen met **Fourier-analyse** (§4.2)
> 2. **Trends** (§4.1)
> 3. **Cycli en autocorrelatie** (§4.3)
> 4. **Andere tijdsgebaseerde features** (§4.4)
>
> Daarnaast: onderzoek de invloed van de features op de voorspelling (§4.6)
> en laat zien hoe jullie iteratief tot de uiteindelijke keuze kwamen
> (§4.7). Dit hoofdstuk en hoofdstuk 5 zijn sterk verweven: ga er
> meermaals heen en terug.

## 4.1 Trend

Een **trend** is een langzame, aanhoudende verandering van het gemiddelde
niveau. Onderzoek of er een trend is (bijv. groei van het bedrijf) en wat
dat betekent voor modellen, vooral omdat de testperiode **ná** de
trainperiode ligt.

In [ ]:
# 1) Lineaire trend op het dagelijkse gemiddelde
t_days = np.arange(len(daily))
fit = stats.linregress(t_days, daily.to_numpy())
print(f"Helling: {fit.slope:.3f} fietsen/uur per dag "
      f"({fit.slope * 365:.1f} per jaar), R2 = {fit.rvalue ** 2:.3f}, "
      f"p = {fit.pvalue:.2g}")

# 2) Groei per maand: 2012 t.o.v. 2011
monthly = (eda.groupby(["year", "month"])[CFG.target].mean().unstack(0))
if {2011, 2012}.issubset(monthly.columns):
    monthly["groei_pct"] = (monthly[2012] / monthly[2011] - 1) * 100
display(monthly.round(1))

# 3) Trendcomponent uit STL naast de lineaire trend
fig, ax = plt.subplots()
daily.plot(ax=ax, alpha=0.3, label="dagelijks gemiddelde")
stl_result.trend.plot(ax=ax, label="STL-trend")
ax.plot(daily.index, fit.intercept + fit.slope * t_days, "--",
        label="lineaire trend")
ax.legend()
ax.set_title("Trend in het gemiddeld aantal verhuurde fietsen")
plt.show()

In [ ]:
# Stationariteit: ADF (H0: eenheidswortel) en KPSS (H0: stationair)
# Dit is relevant voor de differentiatieparameter d in (S)ARIMA (§5.4).
series_tests = {"cnt": y_reg, "cnt (1e verschil)": y_reg.diff().dropna(),
                "cnt (24u-verschil)": y_reg.diff(24).dropna()}
rows = []
for label, series in series_tests.items():
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        rows.append({"reeks": label,
                     "ADF p-waarde": adfuller(series)[1],
                     "KPSS p-waarde": kpss(series, regression="c")[1]})
display(pd.DataFrame(rows).round(4))

> **TODO (team):** is er een trend? Hoe sterk, en is die lineair of
> afvlakkend? Wat betekent dit voor boom-gebaseerde modellen (die kunnen
> **niet extrapoleren** buiten het bereik van de trainingsdata)? Welke
> trendfeatures ga je proberen (bijv. lineaire tijdsindex, jaar-dummy,
> trend uit een hybride model)? Interpreteer de ADF/KPSS-uitkomsten.

## 4.2 Seizoenspatronen met Fourier-analyse

Een **seizoenspatroon** is een regelmatig terugkerend patroon met een
vaste periode. Met de **Fourier-transformatie** (FFT) ontleed je de reeks in
sinusgolven en zie je bij welke periodes de meeste "energie" zit.

In [ ]:
def fourier_spectrum(series, spacing_hours=1.0):
    """Bereken het amplitudespectrum van een gedetrende, regelmatige reeks.

    Geeft een DataFrame terug met de periode (in uren) en de amplitude.
    """
    values = signal.detrend(series.to_numpy(dtype=float))
    window = np.hanning(len(values))          # beperkt spectraal lekken
    amplitude = 2 * np.abs(rfft(values * window)) / window.sum()
    freqs = rfftfreq(len(values), d=spacing_hours)
    spectrum = pd.DataFrame({"period_hours": 1 / freqs[1:],
                             "amplitude": amplitude[1:]})
    return spectrum


def top_peaks(spectrum, n_peaks=8):
    """Geef de n hoogste lokale pieken uit een spectrum."""
    peaks, _ = signal.find_peaks(spectrum["amplitude"].to_numpy())
    top = spectrum.iloc[peaks].nlargest(n_peaks, "amplitude")
    return top.reset_index(drop=True)


spectrum = fourier_spectrum(y_reg)
fig, ax = plt.subplots()
ax.plot(spectrum["period_hours"], spectrum["amplitude"])
ax.set(xscale="log", xlim=(2, 24 * 400), xlabel="Periode (uren, log)",
       ylabel="Amplitude", title="Amplitudespectrum van cnt")
for period, label in [(12, "12u"), (24, "dag"), (168, "week"),
                      (8766, "jaar")]:
    ax.axvline(period, color="red", ls=":", alpha=0.6)
    ax.text(period, ax.get_ylim()[1] * 0.9, label, color="red")
plt.show()
display(top_peaks(spectrum).round(2))

> **TODO (team):** welke periodes vind je (24u, 12u, 168u, jaar)? Waarom
> is de jaarperiode lastig te schatten met maar ~2 jaar data (frequentie-
> resolutie)? Waarom is het raadzaam eerst te **detrenden**? Leg uit wat
> amplitude en periode betekenen.

### Fourier-features

Elk seizoen met periode $P$ kan benaderd worden met $K$ sinus/cosinus-paren:
$\sum_{k=1}^{K}\left[a_k \sin\left(\tfrac{2\pi k t}{P}\right) +
b_k \cos\left(\tfrac{2\pi k t}{P}\right)\right]$. Een hogere $K$ geeft een
flexibeler (maar mogelijk overgefit) patroon. Onderzoek hieronder welke $K$
zinvol is.

In [ ]:
def fourier_terms(index, period, order):
    """Maak sin/cos-termen voor een seizoensperiode (in uren)."""
    hours = np.asarray((index - EPOCH) / pd.Timedelta(hours=1), dtype=float)
    columns = {}
    for k in range(1, order + 1):
        angle = 2 * np.pi * k * hours / period
        columns[f"sin_{period}_{k}"] = np.sin(angle)
        columns[f"cos_{period}_{k}"] = np.cos(angle)
    return pd.DataFrame(columns, index=index)


# Hoeveel van de variatie verklaren K harmonischen? (alleen indicatief)
rows = []
for period, orders in [(24, [1, 2, 3, 4, 6, 8]),
                       (168, [1, 2, 3, 4, 6, 8, 12])]:
    for order in orders:
        X_f = fourier_terms(y_reg.index, period, order)
        r2 = LinearRegression().fit(X_f, y_reg).score(X_f, y_reg)
        rows.append({"periode": period, "K": order, "R2": r2})
display(pd.DataFrame(rows).pivot(index="K", columns="periode",
                                 values="R2").round(3))

# Visualisatie: gemiddeld dagprofiel versus Fourier-fit
profile = eda.groupby("hour")[CFG.target].mean()
fig, ax = plt.subplots()
ax.plot(profile.index, profile.to_numpy(), "k-", lw=3, label="gemiddeld")
for order in (1, 2, 4):
    X_f = fourier_terms(y_reg.index, 24, order)
    fitted = pd.Series(LinearRegression().fit(X_f, y_reg).predict(X_f),
                       index=y_reg.index)
    ax.plot(fitted.groupby(fitted.index.hour).mean(), label=f"K={order}")
ax.set(xlabel="Uur van de dag", ylabel="cnt",
       title="Dagprofiel: Fourier-benadering voor verschillende K")
ax.legend()
plt.show()

> **TODO (team):** welke perioden en welke $K$ kies je, en waarom? Waar zie
> je afnemende meeropbrengst? Waarom is een dagpatroon voor **werkdagen en
> weekenddagen** verschillend (en dus een interactie nodig)?

## 4.3 Cycli en autocorrelatie

Een **cyclus** is een terugkerend patroon **zonder vaste periode** (bijv.
weersystemen die enkele dagen duren); **autocorrelatie** meet hoe sterk de
reeks samenhangt met zichzelf op eerdere tijdstippen (lags).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(y_reg, lags=24 * 8, ax=axes[0])
axes[0].set_title("ACF van cnt (lags in uren)")
plot_pacf(y_reg, lags=48, method="ywm", ax=axes[1])
axes[1].set_title("PACF van cnt")
plt.show()

# Wat blijft er over nadat het vaste uur x weekdag-patroon is weggehaald?
pattern = y_reg.groupby([y_reg.index.dayofweek, y_reg.index.hour]
                        ).transform("mean")
resid_after_seasonality = y_reg - pattern
fig, ax = plt.subplots()
plot_acf(resid_after_seasonality, lags=24 * 7, ax=ax)
ax.set_title("ACF van cnt na verwijderen van uur x weekdag-patroon")
plt.show()

> **TODO (team):** interpreteer de ACF/PACF (pieken op lag 24 en 168? traag
> uitdovend?). Wat blijft er over aan autocorrelatie na het weghalen van het
> seizoenspatroon, en waardoor kan dat komen (weer, trend)? Wat zeggen
> ACF/PACF over de orde $(p, d, q)(P, D, Q)_s$ van een SARIMA(X)-model?
>
> **Let op – lags van de target:** de testperiode is 19 dagen (456 uur) lang
> en `cnt` is daar **onbekend**. Een feature als `cnt` van 1 uur geleden is
> voor het grootste deel van de testperiode dus niet beschikbaar (tenzij je
> recursief voorspelt en fouten opstapelt). Onderbouw je keuze hierover;
> weerfeatures met lags/rolling zijn wél bekend in de testset.

## 4.4 Andere tijdsgebaseerde features

Onderzoek extra features, bijvoorbeeld: werkdag/weekend, spitsuren, dag voor/
na een feestdag, vertraagd of voortschrijdend weer (gisteren regen, gemiddelde
temperatuur laatste 24 uur), en interacties (uur x werkdag).

In [ ]:
# Voorbeeld 1: effect van de (verborgen) vertraging van het weer.
# Correlatie tussen het residu en temperatuur met een vertraging van L uur.
temp_reg = full_reg.loc[y_reg.index, "temp"]
lag_corr = {lag: resid_after_seasonality.corr(temp_reg.shift(lag))
            for lag in range(0, 49, 3)}
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar([str(k) for k in lag_corr], list(lag_corr.values()))
ax.set(xlabel="Vertraging temperatuur (uren)", ylabel="correlatie",
       title="Correlatie residu en vertraagde temperatuur")
plt.show()

# Voorbeeld 2: weekdag-effect in het ochtend-/avondspitsuur
rush = eda[eda["hour"].isin([8, 17, 18])]
sns.boxplot(data=rush, x="dayofweek", y=CFG.target, hue="hour",
            palette="tab10")
plt.title("cnt in spitsuren per weekdag")
plt.show()

> **TODO (team):** welke extra features zijn (niet) zinvol, en waarom?
> Voeg eigen ideeën toe aan `FeatureEngineer` (§4.5). Voorbeelden om zelf te
> onderzoeken: lags/rolling van het weer, aantal uren sinds regen,
> temperatuurklassen, `atemp - temp`, jaar-dummy, schoolvakanties,
> interacties tussen weer en werkdag.

## 4.5 De `FeatureEngineer`-klasse

Alle features worden in één klasse gebouwd. Via `FeatureConfig` zet je
**groepen features** aan/uit, zodat je de invloed per groep kunt meten
(§4.6). De klasse gebruikt **uitsluitend** kolommen die ook in de testset
bekend zijn (geen target-lags), zodat er geen datalek (*data leakage*) is.

In [ ]:
@dataclass
class FeatureConfig:
    """Schakelaars voor groepen features."""

    calendar: bool = True           # uur, weekdag, maand, werkdag, ...
    cyclic: bool = True             # sin/cos van uur, weekdag, maand
    trend: bool = True              # lineaire trend (in jaren)
    fourier: bool = True            # Fourier-termen per seizoensperiode
    weather_rolling: bool = True    # voortschrijdend gemiddelde weer
    interactions: bool = True       # dagpatroon x werkdag
    extra: bool = True              # spitsuur, vlak voor/na vakantie, ...
    fourier_orders: dict = field(
        default_factory=lambda: {24: 4, 168: 3, 8766: 2})
    rolling_windows: tuple = (3, 24)


class FeatureEngineer:
    """Bouwt (tijdserie)features op basis van de opgeschoonde data."""

    BASE_COLS = ["holiday", "weathersit", "temp", "atemp", "hum",
                 "windspeed"]

    def __init__(self, config=None):
        self.config = config or FeatureConfig()
        self.groups_ = {}

    def transform(self, df):
        """Bouw de featurematrix; `df` heeft een DatetimeIndex (gesorteerd)."""
        cfg = self.config
        feats = df[self.BASE_COLS].astype(float).copy()
        self.groups_ = {"basis": list(self.BASE_COLS)}
        builders = [("calendar", cfg.calendar, self._calendar),
                    ("cyclic", cfg.cyclic, self._cyclic),
                    ("trend", cfg.trend, self._trend),
                    ("fourier", cfg.fourier, self._fourier),
                    ("weather_rolling", cfg.weather_rolling,
                     self._weather_rolling),
                    ("interactions", cfg.interactions, self._interactions),
                    ("extra", cfg.extra, self._extra)]
        for name, enabled, builder in builders:
            if enabled:
                block = builder(df)
                self.groups_[name] = list(block.columns)
                feats = feats.join(block)
        return feats

    @staticmethod
    def _workingday(df):
        """1 op doordeweekse dagen die geen vakantiedag zijn."""
        idx = df.index
        flag = (idx.dayofweek < 5) & (df["holiday"].to_numpy() == 0)
        return pd.Series(flag.astype(int), index=idx)

    def _calendar(self, df):
        idx = df.index
        out = pd.DataFrame(index=idx)
        out["hour"] = idx.hour
        out["dayofweek"] = idx.dayofweek
        out["month"] = idx.month
        out["dayofyear"] = idx.dayofyear
        out["is_weekend"] = (idx.dayofweek >= 5).astype(int)
        out["workingday"] = self._workingday(df)
        return out

    @staticmethod
    def _cyclic(df):
        idx = df.index
        out = pd.DataFrame(index=idx)
        for name, values, period in [("hour", idx.hour, 24),
                                     ("dow", idx.dayofweek, 7),
                                     ("month", idx.month, 12)]:
            angle = 2 * np.pi * np.asarray(values) / period
            out[f"{name}_sin"] = np.sin(angle)
            out[f"{name}_cos"] = np.cos(angle)
        return out

    @staticmethod
    def _trend(df):
        years = (df.index - EPOCH) / pd.Timedelta(days=365.25)
        return pd.DataFrame({"trend": np.asarray(years, dtype=float)},
                            index=df.index)

    def _fourier(self, df):
        blocks = [fourier_terms(df.index, period, order)
                  for period, order in self.config.fourier_orders.items()]
        return pd.concat(blocks, axis=1)

    def _weather_rolling(self, df):
        out = pd.DataFrame(index=df.index)
        for column in ("temp", "hum", "windspeed"):
            for window in self.config.rolling_windows:
                out[f"{column}_roll{window}h"] = (
                    df[column].rolling(f"{window}h", min_periods=1).mean())
        return out

    def _interactions(self, df):
        daily = fourier_terms(df.index, 24, 3)
        return daily.mul(self._workingday(df), axis=0).add_prefix("work_x_")

    def _extra(self, df):
        idx = df.index
        hour = idx.hour
        work = self._workingday(df).to_numpy()
        out = pd.DataFrame(index=idx)
        out["rush_hour"] = ((work == 1)
                            & np.isin(hour, [7, 8, 9, 16, 17, 18, 19])
                            ).astype(int)
        out["night"] = (hour <= 5).astype(int)
        holiday_per_day = df["holiday"].groupby(idx.normalize()).max()
        day = pd.Series(idx.normalize(), index=idx)
        out["day_before_holiday"] = ((day + pd.Timedelta(days=1))
                                     .map(holiday_per_day)
                                     .fillna(0).astype(int))
        out["day_after_holiday"] = ((day - pd.Timedelta(days=1))
                                    .map(holiday_per_day)
                                    .fillna(0).astype(int))
        out["atemp_minus_temp"] = df["atemp"] - df["temp"]
        return out


def build_datasets(feature_config=None):
    """Bouw X_train, y_train en X_test met exact dezelfde pipeline."""
    engineer = FeatureEngineer(feature_config)
    feats = engineer.transform(full_clean)
    X_train = feats.loc[train_clean.index]
    X_test = feats.loc[test_clean.index]
    y_train = train_clean[CFG.target]
    return X_train, y_train, X_test, engineer


X_all, y_all, X_test_all, engineer_all = build_datasets(FeatureConfig())
print("X_train:", X_all.shape, "| X_test:", X_test_all.shape)
print({group: len(cols) for group, cols in engineer_all.groups_.items()})
print("NaN in features:", int(X_all.isna().sum().sum()),
      int(X_test_all.isna().sum().sum()))

## 4.6 Invloed van de features op de voorspelling

Om de invloed van features te meten bouw je de featureset **stapsgewijs op**
(*ablation*) en vergelijk je de cross-validated RMSE. Dit doen we met een
snel lineair model (Ridge) en een boomgebaseerd model, omdat ze heel
verschillend reageren op bijv. cyclische codering.

In [ ]:
def make_config(**enabled):
    """Maak een FeatureConfig waarin alleen de genoemde groepen aan staan."""
    flags = dict(calendar=False, cyclic=False, trend=False, fourier=False,
                 weather_rolling=False, interactions=False, extra=False)
    flags.update(enabled)
    return FeatureConfig(**flags)


ABLATION_STEPS = {
    "0_basis": make_config(),
    "1_+calendar": make_config(calendar=True),
    "2_+cyclic": make_config(calendar=True, cyclic=True),
    "3_+trend": make_config(calendar=True, cyclic=True, trend=True),
    "4_+fourier": make_config(calendar=True, cyclic=True, trend=True,
                              fourier=True),
    "5_+weather_rolling": make_config(
        calendar=True, cyclic=True, trend=True, fourier=True,
        weather_rolling=True),
    "6_+interactions": make_config(
        calendar=True, cyclic=True, trend=True, fourier=True,
        weather_rolling=True, interactions=True),
    "7_+extra (alles)": FeatureConfig(),
}

quick_models = {
    "ridge": Pipeline([("scaler", StandardScaler()),
                       ("model", Ridge(alpha=1.0))]),
    "hgb": HistGradientBoostingRegressor(random_state=CFG.random_state),
}

rows = []
for step, feature_cfg in ABLATION_STEPS.items():
    X_step, y_step, _, _ = build_datasets(feature_cfg)
    for model_name, model in quick_models.items():
        scores = cross_val_rmse(model, X_step, y_step)
        LOG.add(f"ablation_{model_name}_{step}", model_name, step,
                scores.mean(), scores.std(), notes="feature-ablation")
        rows.append({"featureset": step, "model": model_name,
                     "n_features": X_step.shape[1],
                     "cv_rmse": scores.mean(), "cv_std": scores.std()})

ablation = pd.DataFrame(rows)
display(ablation.pivot(index="featureset", columns="model",
                       values="cv_rmse").round(1))

In [ ]:
# "Drop-one-group": wat gebeurt er als je één groep uit de volledige set haalt?
rows = []
full_cfg = FeatureConfig()
for group in ["calendar", "cyclic", "trend", "fourier", "weather_rolling",
              "interactions", "extra"]:
    X_drop, y_drop, _, _ = build_datasets(replace(full_cfg, **{group: False}))
    for model_name, model in quick_models.items():
        scores = cross_val_rmse(model, X_drop, y_drop)
        rows.append({"zonder_groep": group, "model": model_name,
                     "cv_rmse": scores.mean()})
drop_one = pd.DataFrame(rows).pivot(index="zonder_groep", columns="model",
                                    values="cv_rmse")
display(drop_one.round(1))

> **TODO (team):** interpreteer beide tabellen. Welke featuregroepen helpen
> welk model? Waarom profiteert een lineair model veel van cyclische/Fourier-
> en interactiefeatures, terwijl een boommodel `hour` direct kan gebruiken?
> Wat gebeurt er met de **trend** bij een boommodel (extrapolatie!)? Let op
> dat de CV-folds in verschillende seizoenen vallen; wees voorzichtig met
> conclusies uit kleine verschillen (kijk ook naar `cv_std`).

## 4.7 Logboek feature-iteraties

> **TODO (team):** houd hier bij hoe jullie iteratief tot de eindkeuze
> kwamen (nieuwe feature → model → CV-resultaat → beslissing). Vul de
> tabel aan na elke iteratie. Het `ExperimentLog` (§2.3) helpt daarbij.

| # | Wijziging (feature/idee) | Hypothese | CV-RMSE vóór | CV-RMSE na | Besluit (houden/schrappen) |
|---|---|---|---|---|---|
| 1 | *TODO* | *TODO* | | | |
| 2 | *TODO* | *TODO* | | | |
| 3 | *TODO* | *TODO* | | | |

**Definitieve featureset (TODO):** kies hieronder de uiteindelijke
configuratie en onderbouw de keuze.

In [ ]:
# TODO: pas aan naar jullie uiteindelijke feature-keuze
FINAL_FEATURE_CFG = FeatureConfig()

X_train, y_train, X_test, ENGINEER = build_datasets(FINAL_FEATURE_CFG)
print("Definitieve features:", X_train.shape[1])
print({group: len(cols) for group, cols in ENGINEER.groups_.items()})

<a id="h5"></a>
# 5. Opdracht 3: Modelleren

[Terug naar boven](#top)

> **Instructie: vereisten Opdracht 3**
> - Modellen die **time series features combineren met reguliere features**:
>   minimaal **5 modellen**, waaronder **lineaire regressie** (met
>   wiskundige uitleg: loss function, regularisatie) en een **ensemble-
>   model** (met wiskundige uitleg). Overige keuzes onderbouwen.
> - **Twee tijdseriemodellen**: één **SARIMA(X)** en één vrij gekozen model;
>   pas toe én leg de werking uit.
> - Een **hybride model** (model 1 voorspelt, model 2 voorspelt de
>   residuen); pas toe en leg de werking uit.
> - **Cross-validation en hyperparameter tuning bij elk model.**
> - Pas elk model toe op de testset en upload de resultaten naar Kaggle.

## 5.1 Validatiestrategie

De trainingsdata loopt tot 30-11-2012 en de testperiode is 1 t/m 19-12-2012:
we voorspellen dus de **toekomst**. Daarom gebruiken we
`TimeSeriesSplit` (*expanding window*): train altijd op het verleden en
valideer op de periode erna, met een validatieperiode die even lang is als
de testperiode (19 dagen).

In [ ]:
def plot_cv_folds(index, cv):
    """Visualiseer train- en validatieperiodes van elke fold."""
    fig, ax = plt.subplots(figsize=(12, 3))
    for fold, (train_idx, val_idx) in enumerate(cv.split(index)):
        ax.barh(fold, index[train_idx[-1]] - index[train_idx[0]],
                left=index[train_idx[0]], color="tab:blue", label="train"
                if fold == 0 else None)
        ax.barh(fold, index[val_idx[-1]] - index[val_idx[0]],
                left=index[val_idx[0]], color="tab:orange",
                label="validatie" if fold == 0 else None)
    ax.set(xlabel="Tijd", ylabel="Fold", title="TimeSeriesSplit")
    ax.legend()
    plt.show()


plot_cv_folds(X_train.index, make_cv())

> **TODO (team):** leg uit waarom een gewone (geshuffelde) K-fold hier tot
> een te optimistische score leidt (denk aan *data leakage* via naburige
> uren) en wat de nadelen zijn van deze aanpak (bijv. weinig data in de
> vroege folds, folds vallen in verschillende seizoenen). Waarom zou je op
> een fold-gemiddelde en de spreiding (std) letten?

## 5.2 Baselines

Een model is pas goed als het een eenvoudige **baseline** verslaat. We
gebruiken (1) het gemiddelde en (2) het gemiddelde per (weekdag, uur).

In [ ]:
class SeasonalProfileBaseline(BaseEstimator, RegressorMixin):
    """Voorspelt het gemiddelde per (weekdag, uur) uit de trainingsdata."""

    def fit(self, X, y):
        key = X["dayofweek"] * 24 + X["hour"]
        self.profile_ = pd.Series(np.asarray(y), index=key.to_numpy()
                                  ).groupby(level=0).mean()
        self.global_mean_ = float(np.mean(y))
        return self

    def predict(self, X):
        key = pd.Series(X["dayofweek"] * 24 + X["hour"], index=X.index)
        return key.map(self.profile_).fillna(self.global_mean_).to_numpy()


baselines = {"baseline_mean": DummyRegressor(strategy="mean"),
             "baseline_profile": SeasonalProfileBaseline()}
for name, model in baselines.items():
    scores = cross_val_rmse(model, X_train, y_train)
    LOG.add(name, name, "n.v.t.", scores.mean(), scores.std())
    print(f"{name:18s} CV-RMSE = {scores.mean():6.1f} (± {scores.std():.1f})")

## 5.3 Reguliere modellen met time series features

We combineren de tijdseriefeatures uit hoofdstuk 4 met de reguliere
features (weer, vakantie) in **ten minste vijf modellen**. In onderstaande
registry staat per model een schatter en een zoekruimte voor tuning.

> **TODO (team):** onderbouw de keuze van elk model (waarom past dit model
> bij deze data? wat zijn sterktes en zwaktes?). Voeg modellen toe of
> vervang ze (bijv. ExtraTrees, SVR, XGBoost/LightGBM, ElasticNet).

In [ ]:
def get_model_registry(random_state=CFG.random_state):
    """Geef per model een tuple: (schatter, zoekruimte voor tuning)."""
    scaler = ("scaler", StandardScaler())
    return {
        "ols": (Pipeline([scaler, ("model", LinearRegression())]), {}),
        "ridge": (
            Pipeline([scaler, ("model", Ridge())]),
            {"model__alpha": loguniform(1e-3, 1e3)}),
        "lasso": (
            Pipeline([scaler, ("model", Lasso(max_iter=20000))]),
            {"model__alpha": loguniform(1e-3, 1e1)}),
        "random_forest": (
            RandomForestRegressor(n_estimators=200, n_jobs=-1,
                                  random_state=random_state),
            {"max_depth": [8, 12, 16, None],
             "min_samples_leaf": randint(1, 10),
             "max_features": [0.3, 0.5, 0.8, 1.0]}),
        "hist_gradient_boosting": (
            HistGradientBoostingRegressor(random_state=random_state),
            {"learning_rate": loguniform(0.02, 0.3),
             "max_iter": randint(100, 600),
             "max_depth": [None, 4, 6, 8],
             "min_samples_leaf": randint(5, 60),
             "l2_regularization": loguniform(1e-3, 10)}),
        "knn": (
            Pipeline([scaler, ("model", KNeighborsRegressor())]),
            {"model__n_neighbors": randint(3, 40),
             "model__weights": ["uniform", "distance"]}),
        "mlp": (
            Pipeline([scaler, ("model", MLPRegressor(
                hidden_layer_sizes=(64, 32), early_stopping=True,
                max_iter=300, random_state=random_state))]),
            {"model__hidden_layer_sizes": [(32,), (64, 32), (128, 64)],
             "model__alpha": loguniform(1e-5, 1e-1),
             "model__learning_rate_init": loguniform(1e-4, 1e-2)}),
    }


def run_default_cv(names, X, y, feature_set="final"):
    """Bereken de CV-score van modellen met standaardinstellingen."""
    registry = get_model_registry()
    rows = []
    for name in names:
        estimator, _ = registry[name]
        scores = cross_val_rmse(estimator, X, y)
        LOG.add(f"{name}_default", name, feature_set, scores.mean(),
                scores.std(), notes="standaardparameters")
        rows.append({"model": name, "cv_rmse": scores.mean(),
                     "cv_std": scores.std()})
    return pd.DataFrame(rows).round(2)

### 5.3.1 Lineaire regressie

> **TODO (team): wiskundige uitleg in eigen woorden, met bronnen.** Neem op:
> 1. Het model: $\hat{y}_i = \beta_0 + \sum_j \beta_j x_{ij}$, in matrixvorm
>    $\hat{\mathbf{y}} = X\boldsymbol{\beta}$.
> 2. De **loss function**: $L(\boldsymbol\beta) = \frac{1}{n}\sum_i
>    (y_i - \hat{y}_i)^2$ (MSE). Hoe hangt dit samen met de Kaggle-RMSE?
> 3. Hoe vind je $\boldsymbol\beta$? Normaalvergelijking
>    $\hat{\boldsymbol\beta} = (X^\top X)^{-1}X^\top \mathbf{y}$ versus
>    gradient descent; wat gaat er mis bij multicollineariteit (`temp` vs
>    `atemp`, sin/cos-termen)?
> 4. **Regularisatie**: Ridge $L + \lambda\lVert\boldsymbol\beta\rVert_2^2$
>    en Lasso $L + \lambda\lVert\boldsymbol\beta\rVert_1$. Effect op
>    bias/variantie, waarom schaal je de features, wat is het verschil in
>    gedrag (krimpen versus nul maken)?
> 5. Beperkingen voor deze data (niet-lineair uurpatroon, interacties).

In [ ]:
display(run_default_cv(["ols", "ridge", "lasso"], X_train, y_train))

# Effect van regularisatie: CV-RMSE en coëfficiënten als functie van alpha
alphas = np.logspace(-3, 3, 13)
cv_rmse_by_alpha, coef_paths = [], []
for alpha in alphas:
    ridge = Pipeline([("scaler", StandardScaler()),
                      ("model", Ridge(alpha=alpha))])
    cv_rmse_by_alpha.append(cross_val_rmse(ridge, X_train, y_train).mean())
    coef_paths.append(ridge.fit(X_train, y_train)["model"].coef_)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].semilogx(alphas, cv_rmse_by_alpha, marker="o")
axes[0].set(xlabel="alpha (lambda)", ylabel="CV-RMSE",
            title="Ridge: invloed van de regularisatiesterkte")
axes[1].semilogx(alphas, np.array(coef_paths))
axes[1].set(xlabel="alpha (lambda)", ylabel="gestandaardiseerde coëfficiënt",
            title="Ridge: coëfficiënten krimpen bij grotere alpha")
plt.show()

> **TODO (team):** interpreteer de plots (bias-variantie-afweging, optimale
> alpha) en de belangrijkste coëfficiënten.

### 5.3.2 Ensemble-modellen (Random Forest en Gradient Boosting)

> **TODO (team): wiskundige uitleg in eigen woorden, met bronnen.**
> - **Bagging / Random Forest:** beslisboom (splitsen op basis van
>   variantiereductie $\Delta = \mathrm{MSE}_{ouder} - \ldots$), bootstrap-
>   samples, gemiddelde van $B$ bomen
>   $\hat{f}(x) = \frac{1}{B}\sum_b T_b(x)$, waarom dit de **variantie**
>   verlaagt (formule met correlatie $\rho$ tussen bomen), rol van
>   `max_features`.
> - **Gradient boosting:** additief model
>   $F_m(x) = F_{m-1}(x) + \nu\, h_m(x)$, waarbij $h_m$ de **negatieve
>   gradiënt** van de loss (bij MSE: de residuen) benadert; rol van learning
>   rate $\nu$, diepte, aantal iteraties en regularisatie.
> - Verschil in bias/variantie tussen bagging en boosting; kunnen bomen
>   **extrapoleren**?

In [ ]:
display(run_default_cv(["random_forest", "hist_gradient_boosting"],
                       X_train, y_train))

### 5.3.3 Overige modellen

> **TODO (team):** leg kort de werking uit van de overige modellen (hier:
> k-nearest neighbours en een neuraal netwerk (MLP)) en onderbouw waarom
> jullie ze gekozen hebben, met bronnen. Vervang ze desgewenst door andere.

In [ ]:
display(run_default_cv(["knn", "mlp"], X_train, y_train))

### 5.3.4 Hyperparameter tuning met cross-validation

We gebruiken `RandomizedSearchCV` met dezelfde tijdsgebaseerde
cross-validation. Willekeurig zoeken is efficiënter dan een volledig grid
bij veel hyperparameters (Bergstra & Bengio, 2012; **TODO:** controleer en
zet de volledige bron in de referentielijst).

> **TODO (team):** pas de zoekruimtes aan, onderbouw ze, en laat zien welke
> hyperparameters het meeste effect hebben. Verhoog `N_ITER` voor de
> definitieve run (let op rekentijd).

In [ ]:
N_ITER = 15            # TODO: verhoog voor de definitieve run
TUNED = {}             # naam -> beste (opnieuw gefit) model
SEARCHES = {}          # naam -> volledig RandomizedSearchCV-object


def tune_model(estimator, param_dist, X, y, cv, n_iter=N_ITER):
    """Tune een model met RandomizedSearchCV en tijdsgebaseerde CV."""
    search = RandomizedSearchCV(
        estimator, param_dist, n_iter=n_iter, cv=cv,
        scoring="neg_root_mean_squared_error", n_jobs=-1,
        random_state=CFG.random_state, refit=True)
    return search.fit(X, y)


def tune_registry(names, X, y, feature_set="final"):
    """Tune alle genoemde modellen en log de resultaten."""
    registry = get_model_registry()
    cv = make_cv()
    for name in names:
        estimator, param_dist = registry[name]
        if not param_dist:           # bijv. OLS heeft geen hyperparameters
            continue
        search = tune_model(estimator, param_dist, X, y, cv)
        best = search.best_index_
        LOG.add(f"{name}_tuned", name, feature_set, -search.best_score_,
                search.cv_results_["std_test_score"][best],
                params=search.best_params_, notes="RandomizedSearchCV")
        TUNED[name] = search.best_estimator_
        SEARCHES[name] = search
        print(f"{name:24s} CV-RMSE = {-search.best_score_:6.1f}  "
              f"{search.best_params_}")


tune_registry(["ridge", "lasso", "random_forest",
               "hist_gradient_boosting", "knn", "mlp"], X_train, y_train)

### 5.3.5 Vergelijking van de reguliere modellen

In [ ]:
results = LOG.to_frame()
regular = results[results["experiment"].str.endswith(("_default", "_tuned"))]
regular = regular.sort_values("cv_rmse")
display(regular[["experiment", "cv_rmse", "cv_std", "params"]].round(2))

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(regular["experiment"], regular["cv_rmse"],
        xerr=regular["cv_std"], color="tab:blue")
ax.axvline(CFG.min_rmse, color="red", ls="--", label="minimumeis Kaggle")
ax.invert_yaxis()
ax.set(xlabel="CV-RMSE (lager is beter)", title="Reguliere modellen")
ax.legend()
plt.show()

> **TODO (team):** welke modellen zijn het beste en waarom? Wat is het
> effect van tuning? Is het verschil groter dan de spreiding tussen folds?

## 5.4 Tijdseriemodellen

Tijdseriemodellen gebruiken de **structuur van de reeks zelf** (seizoen,
autocorrelatie). Ze werken op een **regelmatige** reeks (`y_reg`, §3.2).
De rekentijd van SARIMAX is hoog, dus we passen het model toe op een
**venster** van de laatste weken en gebruiken minder folds.

In [ ]:
TS_CV = make_cv(n_splits=3)      # minder folds vanwege rekentijd

# Exogene variabelen voor SARIMAX (ook bekend in de testperiode)
ts_features = FeatureEngineer(
    make_config(calendar=True, trend=True, fourier=True)
).transform(full_reg)
SARIMAX_EXOG = (["temp", "hum", "windspeed", "workingday", "trend"]
                + [c for c in ts_features.columns
                   if c.startswith(("sin_168", "cos_168"))])
exog_reg = ts_features.loc[y_reg.index, SARIMAX_EXOG]
future_index = full_reg.index[full_reg.index > train_end]
exog_future = ts_features.loc[future_index, SARIMAX_EXOG]
print("Exogene variabelen:", SARIMAX_EXOG)
print("Voorspelhorizon testperiode (uren):", len(future_index))

### 5.4.1 SARIMA(X)

> **TODO (team): uitleg in eigen woorden, met bronnen.** Neem op:
> - Notatie $\mathrm{SARIMA}(p,d,q)(P,D,Q)_s$ en de betekenis van elke
>   parameter; backshift-operator $B$; AR-, MA- en integratiedeel
>   (differentiëren voor stationariteit, zie ADF/KPSS in §4.1).
> - Het seizoensdeel met periode $s = 24$ (en waarom niet ook $s = 168$).
> - De **X**: lineaire invloed van exogene variabelen $\beta^\top x_t$ met
>   ARMA-fouten (regressie met SARIMA-residuen).
> - Hoe kies je de orde (ACF/PACF uit §4.3, AIC, cross-validation)?
> - Beperkingen: één seizoensperiode, hoge rekentijd, onzekerheid neemt toe
>   bij een horizon van 456 uur.

In [ ]:
def sarimax_forecast(y_hist, exog_hist, exog_next, order, seasonal_order,
                     window=24 * 7 * 6):
    """Fit een SARIMAX op de laatste `window` uren en voorspel de toekomst."""
    y_win = y_hist.iloc[-window:]
    exog_win = exog_hist.iloc[-window:]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = SARIMAX(y_win, exog=exog_win, order=order,
                        seasonal_order=seasonal_order,
                        enforce_stationarity=False,
                        enforce_invertibility=False)
        result = model.fit(disp=False, maxiter=100)
        forecast = result.forecast(steps=len(exog_next), exog=exog_next)
    return np.clip(np.asarray(forecast), 0, None)


def make_sarimax_fold(order, seasonal_order, window):
    """Maak een fit_predict-functie voor `ts_cv_rmse`."""
    def fit_predict(train_idx, val_idx):
        return sarimax_forecast(
            y_reg.iloc[train_idx], exog_reg.iloc[train_idx],
            exog_reg.iloc[val_idx], order, seasonal_order, window)
    return fit_predict


# TODO: onderbouw de kandidaten met ACF/PACF en AIC; breid het grid uit
SARIMAX_GRID = [((1, 0, 0), (1, 0, 0, 24)),
                ((2, 0, 1), (1, 0, 0, 24)),
                ((1, 0, 1), (1, 0, 1, 24))]
SARIMAX_WINDOW = 24 * 7 * 6        # TODO: probeer ook andere vensters

rows = []
for order, seasonal_order in SARIMAX_GRID:
    scores = ts_cv_rmse(
        make_sarimax_fold(order, seasonal_order, SARIMAX_WINDOW),
        y_reg, TS_CV)
    rows.append({"order": order, "seasonal_order": seasonal_order,
                 "cv_rmse": scores.mean(), "cv_std": scores.std()})
    print(rows[-1])
sarimax_results = pd.DataFrame(rows).sort_values("cv_rmse")
display(sarimax_results.round(2))

In [ ]:
best_sarimax = sarimax_results.iloc[0]
LOG.add("sarimax_tuned", "sarimax", "exog+fourier_168",
        best_sarimax["cv_rmse"], best_sarimax["cv_std"],
        params={"order": best_sarimax["order"],
                "seasonal_order": best_sarimax["seasonal_order"],
                "window": SARIMAX_WINDOW}, notes="3 folds")

# Voorspelling voor de testperiode (fit op einde van de volledige trainreeks)
sarimax_pred = pd.Series(
    sarimax_forecast(y_reg, exog_reg, exog_future, best_sarimax["order"],
                     best_sarimax["seasonal_order"], SARIMAX_WINDOW),
    index=future_index).reindex(test_clean.index)
print("NaN in voorspelling:", int(sarimax_pred.isna().sum()))

> **TODO (team):** bespreek de uitkomsten: hoe presteert SARIMAX t.o.v. de
> reguliere modellen? Wat zeggen de residuen (ACF van de residuen)? Waarom
> degradeert de voorspelling naarmate de horizon langer wordt?

### 5.4.2 Tweede tijdseriemodel (vrije keuze): Holt-Winters

Hier is gekozen voor Holt-Winters (exponentiële afvlakking met trend en
seizoen). **TODO:** je mag een ander model kiezen (bijv. Prophet, TBATS,
STL + ETS, UnobservedComponents). Onderbouw je keuze.

> **TODO (team): uitleg in eigen woorden, met bronnen:** niveau-, trend- en
> seizoenscomponent, de afvlakkingsparameters $\alpha,\beta,\gamma$,
> additief versus multiplicatief, gedempte trend, en waarom dit model geen
> exogene variabelen (weer) gebruikt.

In [ ]:
def holt_winters_forecast(y_hist, steps, trend, damped,
                          window=24 * 7 * 8):
    """Fit Holt-Winters (seizoen 24u) en voorspel `steps` uur vooruit."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model = ExponentialSmoothing(
            y_hist.iloc[-window:], trend=trend,
            damped_trend=damped if trend else False, seasonal="add",
            seasonal_periods=24, initialization_method="estimated")
        forecast = model.fit().forecast(steps)
    return np.clip(np.asarray(forecast), 0, None)


def make_hw_fold(trend, damped, window):
    """Maak een fit_predict-functie voor `ts_cv_rmse`."""
    def fit_predict(train_idx, val_idx):
        return holt_winters_forecast(y_reg.iloc[train_idx], len(val_idx),
                                     trend, damped, window)
    return fit_predict


HW_GRID = [(None, False, 24 * 7 * 8), ("add", False, 24 * 7 * 8),
           ("add", True, 24 * 7 * 8), ("add", True, 24 * 7 * 4)]
rows = []
for trend, damped, window in HW_GRID:
    scores = ts_cv_rmse(make_hw_fold(trend, damped, window), y_reg, TS_CV)
    rows.append({"trend": trend, "damped": damped, "window": window,
                 "cv_rmse": scores.mean(), "cv_std": scores.std()})
hw_results = pd.DataFrame(rows).sort_values("cv_rmse")
display(hw_results.round(2))

best_hw = hw_results.iloc[0]
LOG.add("holt_winters_tuned", "holt_winters", "geen (univariaat)",
        best_hw["cv_rmse"], best_hw["cv_std"],
        params=best_hw[["trend", "damped", "window"]].to_dict(),
        notes="3 folds")
holt_winters_pred = pd.Series(
    holt_winters_forecast(y_reg, len(future_index), best_hw["trend"],
                          best_hw["damped"], int(best_hw["window"])),
    index=future_index).reindex(test_clean.index)

> **TODO (team):** vergelijk beide tijdseriemodellen met elkaar en met de
> reguliere modellen. Wat is het voordeel van exogene variabelen?

## 5.5 Hybride model

Een **hybride model** combineert twee modellen: model 1 (hier: een
lineair model op trend + Fourier-seizoen) legt de **systematische structuur**
vast en model 2 (hier: gradient boosting) voorspelt de **residuen** van
model 1 op basis van alle features. De eindvoorspelling is de som:
$\hat{y}_t = \hat{f}_1(x_t) + \hat{f}_2(x_t)$.

> **TODO (team): uitleg in eigen woorden, met bronnen.** Waarom helpt deze
> combinatie? (Denk aan: bomen kunnen niet extrapoleren; een lineair model
> kan dat wel voor de trend.) Waarom moeten de residuen van model 1 op de
> **trainingsdata** gebruikt worden, en wanneer is er een risico op
> overfitting (als model 1 zeer flexibel is, gebruik dan out-of-fold
> residuen)? Wat zou een alternatief zijn (bijv. SARIMAX + boosting)?

In [ ]:
class HybridRegressor(BaseEstimator, RegressorMixin):
    """Model 1 voorspelt de structuur, model 2 de residuen van model 1."""

    def __init__(self, base_model, residual_model, base_cols,
                 residual_cols=None):
        self.base_model = base_model
        self.residual_model = residual_model
        self.base_cols = base_cols
        self.residual_cols = residual_cols

    def fit(self, X, y):
        y = np.asarray(y, dtype=float)
        self.residual_cols_ = (list(X.columns) if self.residual_cols is None
                               else list(self.residual_cols))
        self.base_model_ = clone(self.base_model).fit(
            X[self.base_cols], y)
        residuals = y - self.base_model_.predict(X[self.base_cols])
        self.residual_model_ = clone(self.residual_model).fit(
            X[self.residual_cols_], residuals)
        return self

    def predict(self, X):
        return (self.base_model_.predict(X[self.base_cols])
                + self.residual_model_.predict(X[self.residual_cols_]))


# Model 1: lineaire trend + Fourier-seizoenen; model 2: boosting op residuen
HYBRID_BASE_COLS = ["trend"] + ENGINEER.groups_.get("fourier", [])
hybrid = HybridRegressor(
    base_model=Ridge(alpha=1.0),
    residual_model=HistGradientBoostingRegressor(
        random_state=CFG.random_state),
    base_cols=HYBRID_BASE_COLS)

scores = cross_val_rmse(hybrid, X_train, y_train)
LOG.add("hybrid_default", "hybrid", "final", scores.mean(), scores.std(),
        notes="Ridge(trend+Fourier) + HGB op residuen")
print(f"Hybride model (standaard): CV-RMSE = {scores.mean():.1f} "
      f"(± {scores.std():.1f})")

In [ ]:
# Hyperparameter tuning van het hybride model (beide deelmodellen)
hybrid_params = {
    "base_model__alpha": loguniform(1e-3, 1e2),
    "residual_model__learning_rate": loguniform(0.02, 0.3),
    "residual_model__max_iter": randint(100, 600),
    "residual_model__max_depth": [None, 4, 6, 8],
    "residual_model__min_samples_leaf": randint(5, 60),
}
hybrid_search = tune_model(hybrid, hybrid_params, X_train, y_train,
                           make_cv())
best = hybrid_search.best_index_
LOG.add("hybrid_tuned", "hybrid", "final", -hybrid_search.best_score_,
        hybrid_search.cv_results_["std_test_score"][best],
        params=hybrid_search.best_params_, notes="RandomizedSearchCV")
TUNED["hybrid"] = hybrid_search.best_estimator_
print(f"Hybride model (getuned): CV-RMSE = {-hybrid_search.best_score_:.1f}")
print(hybrid_search.best_params_)

> **TODO (team):** verslaat het hybride model zijn losse deelmodellen
> (alleen Ridge, alleen boosting)? Laat dit zien met een plot van
> voorspellingen/residuen over de tijd. Probeer eventueel een andere
> combinatie (bijv. SARIMAX als model 1).

## 5.6 Voorspellen op de testset en uploaden naar Kaggle

Elk definitief model wordt op **alle trainingsdata** opnieuw gefit en
voorspelt daarna de testperiode. Voor Kaggle is een csv nodig in het
formaat van `sample_submission.csv`.

In [ ]:
class SubmissionWriter:
    """Schrijft voorspellingen weg in het Kaggle-submissionformaat."""

    def __init__(self, config):
        self.config = config
        path = config.data_dir / config.sample_file
        self.sample = pd.read_csv(path, dtype=str)    # originele notatie
        self.keys = pd.DatetimeIndex(
            pd.to_datetime(self.sample[config.date_col]))

    def write(self, predictions, name):
        """Schrijf een Series (DatetimeIndex) weg als submission-csv."""
        aligned = predictions.reindex(self.keys)
        if aligned.isna().any():
            raise ValueError("Voorspellingen ontbreken voor sommige uren.")
        out = self.sample[[self.config.date_col]].copy()
        out[self.config.target] = np.clip(aligned.to_numpy(), 0, None)
        stamp = datetime.now().strftime("%Y%m%d_%H%M")
        path = self.config.submission_dir / f"{stamp}_{name}.csv"
        out.to_csv(path, index=False)
        return path


def predict_test(model, X_fit, y_fit, X_new):
    """Fit op alle trainingsdata en voorspel de testperiode."""
    fitted = clone(model).fit(X_fit, y_fit)
    return pd.Series(np.clip(fitted.predict(X_new), 0, None),
                     index=X_new.index)


writer = SubmissionWriter(CFG)
PREDICTIONS = {}
for name, model in TUNED.items():
    PREDICTIONS[f"{name}_tuned"] = predict_test(
        model, X_train, y_train, X_test)
PREDICTIONS["sarimax_tuned"] = sarimax_pred
PREDICTIONS["holt_winters_tuned"] = holt_winters_pred

for experiment, preds in PREDICTIONS.items():
    path = writer.write(preds, experiment)
    print(f"{experiment:28s} -> {path.name}")

### Kaggle-scores bijhouden

Upload elk bestand uit de map `submissions/` op Kaggle en vul hieronder de
**public score (RMSE)** in. Dit gebruiken we in hoofdstuk 6 om de
verbetering van de score te laten zien.

> **Let op:** de ingeleverde voorspellingen moeten het resultaat zijn van
> jullie **eigen getrainde modellen**; andere modellen of online gevonden
> labels leiden tot diskwalificatie.

In [ ]:
# TODO: vul de Kaggle public scores in (sleutel = experimentnaam)
KAGGLE_SCORES = {
    # "ridge_tuned": 95.2,
    # "hist_gradient_boosting_tuned": 62.7,
}
for experiment, score in KAGGLE_SCORES.items():
    LOG.set_kaggle_score(experiment, score)
LOG.save()

| Submission | Datum | Bestand | Model + features | Kaggle RMSE | Opmerking |
|---|---|---|---|---|---|
| 1 | *TODO* | *TODO* | *TODO* | *TODO* | *TODO* |
| 2 | *TODO* | *TODO* | *TODO* | *TODO* | *TODO* |

<a id="h6"></a>
# 6. Bevindingen

[Terug naar boven](#top)

> **Instructie: vereisten Bevindingen**
> - Samenvatting van de uitkomsten van het modelleren.
> - Beknopt overzicht van de resultaten.
> - Welk model presteert het best, en bij welke parameters?
> - Kaggle-scores en de verbetering ervan over de tijd.

In [ ]:
results = LOG.to_frame()
modelling = results[~results["experiment"].str.startswith("ablation_")]
overview = modelling.sort_values("cv_rmse")[
    ["experiment", "model", "feature_set", "cv_rmse", "cv_std",
     "kaggle_rmse"]].round(2)
display(overview.head(15))

best_row = modelling.loc[modelling["cv_rmse"].idxmin()]
print("Beste model volgens CV:", best_row["experiment"])
print("CV-RMSE:", round(best_row["cv_rmse"], 2))
print("Parameters:", best_row["params"])

In [ ]:
# Verbetering van de Kaggle-score in de tijd
scored = results.dropna(subset=["kaggle_rmse"]).sort_values("timestamp")
if scored.empty:
    print("Nog geen Kaggle-scores ingevuld (zie KAGGLE_SCORES in §5.6).")
else:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(range(1, len(scored) + 1), scored["kaggle_rmse"], marker="o")
    for pos, name in enumerate(scored["experiment"], start=1):
        ax.annotate(name, (pos, scored["kaggle_rmse"].iloc[pos - 1]),
                    rotation=30, fontsize=8)
    ax.axhline(CFG.min_rmse, color="red", ls="--", label="minimumeis (80)")
    ax.set(xlabel="Submission (chronologisch)", ylabel="Kaggle RMSE",
           title="Verloop van de Kaggle-score")
    ax.legend()
    plt.show()

> **TODO (team):** schrijf de bevindingen uit:
> 1. **Samenvatting** van de modelleerresultaten (reguliere modellen,
>    tijdseriemodellen, hybride model).
> 2. **Beste model** en de bijbehorende (getunede) parameters.
> 3. **Kaggle-scores** en de stappen die de score verbeterden (welke
>    feature/modelwijziging gaf hoeveel winst?).
> 4. Verschil tussen **CV-score en Kaggle-score**: waar komt dat vandaan
>    (andere periode/seizoen, december-effect, toeval)?
> 5. Waar presteren de modellen slecht (bijv. specifieke uren/dagen)?
>    Voeg een residu-analyse toe.

In [ ]:
# Voorbeeld residu-analyse op de laatste CV-fold (uit te breiden)
best_name = "hist_gradient_boosting"        # TODO: kies het beste model
last_train, last_val = list(make_cv().split(X_train))[-1]
model_fold = clone(TUNED[best_name]).fit(X_train.iloc[last_train],
                                         y_train.iloc[last_train])
pred_fold = pd.Series(np.clip(model_fold.predict(X_train.iloc[last_val]),
                              0, None), index=X_train.index[last_val])
actual_fold = y_train.iloc[last_val]

fig, axes = plt.subplots(2, 1, figsize=(12, 7))
actual_fold.plot(ax=axes[0], label="werkelijk")
pred_fold.plot(ax=axes[0], label="voorspeld")
axes[0].legend()
axes[0].set_title(f"{best_name}: validatieperiode van de laatste fold")
(actual_fold - pred_fold).groupby(actual_fold.index.hour).mean().plot(
    kind="bar", ax=axes[1])
axes[1].set(xlabel="Uur", ylabel="gemiddeld residu",
            title="Gemiddeld residu per uur van de dag")
plt.tight_layout()
plt.show()

<a id="h7"></a>
# 7. Opdracht 4: Conclusie en aanbevelingen

[Terug naar boven](#top)

> **Instructie: vereisten Opdracht 4**
> - Welke features hebben de meeste invloed op de voorspellingen?
> - In hoeverre sluit dit aan op de EDA?
> - Welk advies geef je op basis hiervan aan de verhuurder?
> - Welk **model** adviseer je en waarom? Is dat per definitie het model
>   met de beste voorspellingen, of zijn er ook andere overwegingen?

## 7.1 Feature-importance

Omdat features sterk samenhangen (bijv. `hour`, `hour_sin`/`hour_cos` en de
Fourier-termen beschrijven dezelfde informatie) kijken we naar de invloed
van **groepen features** (gezamenlijk geschud) én naar individuele features.

In [ ]:
def grouped_permutation_importance(model, X, y, groups, n_repeats=10,
                                   seed=CFG.random_state):
    """Schud groepen kolommen gezamenlijk en meet de stijging in RMSE."""
    rng = np.random.default_rng(seed)
    baseline = rmse(y, np.clip(model.predict(X), 0, None))
    rows = []
    for group, cols in groups.items():
        increases = []
        for _ in range(n_repeats):
            shuffled = X.copy()
            order = rng.permutation(len(X))
            shuffled[cols] = X[cols].to_numpy()[order]
            pred = np.clip(model.predict(shuffled), 0, None)
            increases.append(rmse(y, pred) - baseline)
        rows.append({"groep": group, "stijging_rmse": np.mean(increases),
                     "std": np.std(increases)})
    return pd.DataFrame(rows).sort_values("stijging_rmse", ascending=False)


# Holdout = laatste 19 dagen; fit op de rest (TODO: kies het beste model)
BEST_MODEL_NAME = "hist_gradient_boosting"
split = len(X_train) - CFG.cv_test_size
model_imp = clone(TUNED[BEST_MODEL_NAME]).fit(X_train.iloc[:split],
                                              y_train.iloc[:split])
X_hold, y_hold = X_train.iloc[split:], y_train.iloc[split:]
print("RMSE op holdout:",
      round(rmse(y_hold, np.clip(model_imp.predict(X_hold), 0, None)), 1))

group_imp = grouped_permutation_importance(model_imp, X_hold, y_hold,
                                           ENGINEER.groups_)
display(group_imp.round(2))

feature_imp = permutation_importance(
    model_imp, X_hold, y_hold, n_repeats=10,
    scoring="neg_root_mean_squared_error", random_state=CFG.random_state)
top = pd.Series(feature_imp.importances_mean,
                index=X_hold.columns).nlargest(15)
fig, ax = plt.subplots(figsize=(8, 5))
top.sort_values().plot(kind="barh", ax=ax)
ax.set(xlabel="stijging RMSE bij schudden",
       title="Top 15 individuele features (permutation importance)")
plt.show()

## 7.2 Conclusie en aansluiting op de EDA

> **TODO (team):** beantwoord onderbouwd:
> - Welke features/featuregroepen hebben de meeste invloed? Komt dat
>   overeen met wat jullie in de EDA (§3.5) zagen? Waar wijkt het af en
>   waarom (bijv. gecorreleerde features verdelen importance)?
> - Wat zeggen de tijdserie-features (trend, seizoen, cycli) over het
>   gedrag van klanten?
> - Hoe robuust zijn deze conclusies? (Permutation importance geeft geen
>   causaal verband en is gevoelig voor gecorreleerde features.)

## 7.3 Advies aan CityBike

> **TODO (team):** vertaal de inzichten naar concrete, uitvoerbare adviezen
> voor de fietsverdeling (bijv. voor welke uren/dagen/weersituaties meer
> fietsen beschikbaar moeten zijn, hoe om te gaan met feestdagen of
> weersverwachtingen, welke aanvullende data zinvol is).
> Benoem ook beperkingen van de analyse.

## 7.4 Modeladvies

> **TODO (team):** welk model adviseren jullie en waarom? Is dat altijd het
> model met de laagste RMSE? Overweeg onder meer:
> - nauwkeurigheid én stabiliteit over folds/periodes;
> - **uitlegbaarheid** richting de verhuurder;
> - reken- en onderhoudskosten (hertrainen, monitoring, complexiteit);
> - afhankelijkheid van **weersvoorspellingen** (die zijn in de praktijk
>   zelf onzeker, in tegenstelling tot de gegeven testdata);
> - vermogen om **trend en nieuwe situaties** te extrapoleren;
> - de **voorspelhorizon** en hoe vaak je kunt/wilt bijsturen.

<a id="h8"></a>
# 8. Referenties en GenAI-gebruik

[Terug naar boven](#top)

## 8.1 Regels voor bronvermelding (samenvatting)

- Verwijs in **tekst en code** naar elke bron, in **APA-stijl**.
- Parafraseer; maximaal 5% van de tekst en 25% van de code mag letterlijk
  (ook bijna-letterlijk) overgenomen zijn, mits gerefereerd.
- Is GenAI de bron? Verwijs dan **ook naar een niet-GenAI bron** die de
  juistheid aantoont.
- **Code:** zet in de codecel een comment `# Prompt 1: <titel>`; in
  markdown de details, bijvoorbeeld:
  `[ChatGPT, 2023. Prompt 1: Typen vulkanen](https://...)`.
- **Tekst:** verwijs na de zin, bijv. `(ChatGPT, 2023. Prompt 1: Typen
  vulkanen)`.
- Bij een gedeeld gesprek: klik op **'Share your name'**, zodat jullie
  gebruikersnaam zichtbaar is voor de docent.

## 8.2 GenAI-gebruikslog

| Prompt nr. | Titel van de chat | Model + versie | Datum | Link naar gesprek | Waar gebruikt (§ / cel) |
|---|---|---|---|---|---|
| 1 | *TODO* | *TODO* | *TODO* | *TODO* | *TODO* |
| 2 | *TODO* | *TODO* | *TODO* | *TODO* | *TODO* |

## 8.3 Referentielijst (APA)

> **TODO (team):** alle bronnen waarnaar in tekst en code is verwezen, op
> alfabetische volgorde. Voorbeelden van het formaat (controleer zelf de
> gegevens):
>
> - Auteur, A. A. (jaar). *Titel van het werk*. Uitgever. URL/DOI
> - ChatGPT, 2023. Prompt 1: Titel. https://chat.openai.com/share/...
> - Documentatie van gebruikte pakketten (scikit-learn, statsmodels, ...)
> - Dataset: bron en beschrijving van de Kaggle-competitie.

1. *TODO*
2. *TODO*

<a id="h9"></a>
# 9. Inleverchecklist

[Terug naar boven](#top)

> **Instructie:** loop dit door vóór het inleveren en verwijder daarna alle
> `> **Instructie**`-blokken.

- [ ] Teamnaam, namen en Kaggle-gebruikersnamen bovenaan ingevuld
- [ ] Alle **TODO**'s opgelost of verwijderd
- [ ] *Kernel -> Restart & Run All* werkt zonder fouten, van boven naar beneden
- [ ] Genummerde hoofdstukken/paragrafen; markdown voor tekst, code voor code
- [ ] Zoveel mogelijk functies/klassen; code volgt **PEP8** (bijv. met
      `flake8` of `pycodestyle`), uniforme naamgeving en comments
- [ ] EDA: datatypes, basisstatistieken, tijdserie-elementen, relaties,
      bevindingen, opschoning onderbouwd
- [ ] Feature engineering: Fourier, trend, cycli/autocorrelatie, overige
      features, invloed per feature onderzocht, iteratief proces zichtbaar
- [ ] Modellen: ≥ 5 reguliere modellen (incl. lineaire regressie + ensemble
      met wiskundige uitleg), SARIMA(X) + tweede tijdseriemodel, hybride
      model, CV + tuning bij **elk** model
- [ ] Alle modellen op de testset toegepast en geüpload naar Kaggle
- [ ] **Kaggle-RMSE ≤ 80**, minimaal één submission, eigen modellen
- [ ] Bevindingen en conclusie/aanbevelingen compleet (incl. modeladvies)
- [ ] Bronvermelding in APA in tekst, figuren en code; GenAI-log compleet
- [ ] Elk teamlid kan alle code en tekst toelichten